# たるこ: Colab + Ollama直結

Google ColabでOllamaを起動し、Cloudflare Quick Tunnelで `http://localhost:11434` を公開します。たるこ側は、表示された `https://....trycloudflare.com` を貼り、形式を `Ollama`、モデルを `gemma4:e2b` にしてください。

この版はFastAPIを挟まず、GitHub PagesからOllama本体の `/api/chat` に直接POSTします。CORSは `OLLAMA_ORIGINS="*"` でOllama側に許可させます。

In [ ]:
# 設定
MODEL = "gemma4:e2b"
OLLAMA_URL = "http://localhost:11434"
OLLAMA_BIN = "/usr/local/bin/ollama"
RUN_PUBLIC_CHAT_TEST = True

In [ ]:
# Ollamaをインストールして起動、モデルをpull
import os
import subprocess
import time

import requests

def run(command, **kwargs):
    print("$", " ".join(command))
    return subprocess.run(command, check=True, **kwargs)

run(["apt-get", "update", "-y"])
run(["apt-get", "install", "-y", "zstd", "curl", "wget"])
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

def ollama_ready():
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=3).ok
    except Exception:
        return False

old_ollama_proc = globals().get("ollama_proc")
if old_ollama_proc is not None and old_ollama_proc.poll() is None:
    old_ollama_proc.terminate()
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(2)

ollama_proc = subprocess.Popen(
    [OLLAMA_BIN, "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy(),
)

deadline = time.time() + 90
while time.time() < deadline:
    if ollama_ready():
        break
    time.sleep(1)
else:
    raise RuntimeError("Ollamaが起動しませんでした")

print("Ollama起動OK")
run([OLLAMA_BIN, "pull", MODEL])

print("モデルをウォームアップします")
warmup = requests.post(
    f"{OLLAMA_URL}/api/chat",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "OKだけ返して"}],
        "stream": False,
        "keep_alive": "30m",
        "options": {"num_predict": 16},
    },
    timeout=300,
)
warmup.raise_for_status()
print("Warm-up:", warmup.json().get("message", {}).get("content", "")[:120])

In [ ]:
# cloudflaredをインストール
import subprocess

run(["wget", "-q", "-O", "cloudflared-linux-amd64.deb", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
run(["dpkg", "-i", "cloudflared-linux-amd64.deb"])
run(["cloudflared", "--version"])

In [ ]:
# Cloudflare Quick Tunnelを起動して公開URLを確認
import re
import threading
import time

import requests

old_tunnel_proc = globals().get("tunnel_proc")
if old_tunnel_proc is not None and old_tunnel_proc.poll() is None:
    print("古いcloudflaredプロセスを停止します")
    old_tunnel_proc.terminate()
    time.sleep(2)

public_url_box = {"url": None}

tunnel_proc = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--protocol",
        "http2",
        "--url",
        OLLAMA_URL,
        "--http-host-header",
        "localhost:11434",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def read_tunnel_output():
    pattern = re.compile(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com")
    for line in tunnel_proc.stdout:
        print(line.rstrip())
        match = pattern.search(line)
        if match and public_url_box["url"] is None:
            public_url_box["url"] = match.group(0)

threading.Thread(target=read_tunnel_output, daemon=True).start()

deadline = time.time() + 120
while time.time() < deadline:
    if public_url_box["url"]:
        break
    if tunnel_proc.poll() is not None:
        raise RuntimeError("cloudflaredが終了しました。上のログを確認してください")
    time.sleep(1)
else:
    raise RuntimeError("trycloudflare URLを取得できませんでした")

PUBLIC_URL = public_url_box["url"]
print("\nたるこに貼るURL:", PUBLIC_URL)
print("形式: Ollama")
print("モデル:", MODEL)

def wait_public_version(timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            response = requests.get(f"{PUBLIC_URL}/api/version", timeout=10)
            if response.ok:
                return response.json()
            print("Public version pending:", response.status_code, response.text[:200])
        except Exception as exc:
            print("Public version pending:", exc)
        time.sleep(2)
    raise RuntimeError("公開URLからOllamaへ到達できませんでした")

print("Public version:", wait_public_version())

cors = requests.options(
    f"{PUBLIC_URL}/api/chat",
    headers={
        "Origin": "https://example.github.io",
        "Access-Control-Request-Method": "POST",
        "Access-Control-Request-Headers": "Content-Type",
    },
    timeout=20,
)
print("CORS preflight:", cors.status_code, cors.headers.get("Access-Control-Allow-Origin"), cors.headers.get("Access-Control-Allow-Methods"))

if RUN_PUBLIC_CHAT_TEST:
    public_chat = requests.post(
        f"{PUBLIC_URL}/api/chat",
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": "日本語で短くOKと返して"}],
            "stream": False,
            "keep_alive": "30m",
            "options": {"num_predict": 16},
        },
        timeout=300,
    )
    public_chat.raise_for_status()
    print("Public chat:", public_chat.json().get("message", {}).get("content", "")[:120])

In [ ]:
# 画像入力のテスト（任意）
# 左のファイル欄か下のアップロードで画像を用意してから実行してください。
import base64
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded))
except Exception:
    IMAGE_PATH = "/content/test.jpg"

image_base64 = base64.b64encode(Path(IMAGE_PATH).read_bytes()).decode("utf-8")

vision_test = requests.post(
    f"{OLLAMA_URL}/api/chat",
    json={
        "model": MODEL,
        "stream": False,
        "messages": [
            {
                "role": "user",
                "content": "この画像を見て、短く説明して。",
                "images": [image_base64],
            }
        ],
    },
    timeout=300,
)
vision_test.raise_for_status()
print(vision_test.json()["message"]["content"])
